# Yoojin — Phase 2: Ratings and Note Status History cleaning

This notebook presents configuration, inspection, cleaning decisions, execution, quality review and validation in the same explanatory format as Peyton's Notes/Note Request notebook. Shared implementation stays in `clean_ratings_status.py`; the notebook and script use the same rules.

Scope: ratings-00006.tsv and noteStatusHistory-00000.tsv only, not a verified complete platform release. This saved execution reruns Ratings only; Status is retained from its earlier completed run. Run in order. Full Ratings processing and independent verification can take substantial time and disk space. Large source and output files stay outside Git.


## 1. Configure paths and scope

Use `PREVIEW_ROWS = 1000` for a first check; set it to `None` for all supplied rows. Preview and full outputs are separate, and existing outputs are never overwritten. The cutoff below is inherited from the original notebook, **not a confirmed common team cutoff**. It adds provisional flags/metrics; no rows are deleted for being after it. Confirm with the team before final analysis. Snapshot information is unknown unless documented.


In [17]:
from pathlib import Path
from datetime import datetime, timezone
import sys, subprocess, json
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), Path.cwd() / 'yoojin_phase2']
WORKFLOW = next((p.resolve() for p in candidates if (p / 'clean_ratings_status.py').is_file()), None)
if WORKFLOW is None:
    raise FileNotFoundError('Open Jupyter in the yoojin_phase2 folder, or add its full path to candidates.')
RAW = Path.home() / 'Downloads'
PREVIEW_ROWS = None # None = full supplied files; 1000 = preview only
CUTOFF = '2026-10-02T23:59:59.999Z'  # inherited; team agreement pending
SNAPSHOT_LABEL = None  # do not infer from file modification dates
ENGINE = 'pandas' if PREVIEW_ROWS is not None else 'auto'
RUN_LABEL = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = WORKFLOW / 'outputs' / ('preview' if PREVIEW_ROWS is not None else 'full') / RUN_LABEL
sys.path.insert(0, str(WORKFLOW))
from clean_ratings_status import expand
FILES = {
    'ratings': expand([str(RAW / 'ratings-00006.tsv')]),
    'status': expand([str(RAW / 'noteStatusHistory-00000.tsv')])
}
print('Mode:', 'PREVIEW ONLY' if PREVIEW_ROWS is not None else 'ALL SUPPLIED ROWS')
print('Output:', OUT)


Mode: ALL SUPPLIED ROWS
Output: /Users/hwang-yujin/dso576/project/yoojin_phase2/outputs/full/20261018T050534418856Z


## 2. Inventory the actual inputs

Check that the matched shards are the intended team scope and snapshot. File names and modification times alone cannot establish release completeness. The cleaning code rejects mismatched shard schemas.

In [18]:
inventory = pd.DataFrame([{'dataset': kind, 'file': str(Path(f)), 'bytes': Path(f).stat().st_size}
                          for kind, files in FILES.items() for f in files])
display(inventory)


,dataset,file,bytes
0,ratings,/Users/hwang-yujin/Downloads/ratings-00006.tsv,5267960910
1,status,/Users/hwang-yujin/Downloads/noteStatusHistory...,888538104


## 3. Inspect structure and missing values

Only the first 1,000 records per file are inspected here. This is a prefix, not a random sample or full-data finding. Original values, blank strings and long IDs are read as strings.

In [19]:
for kind, files in FILES.items():
    for file in files:
        sample = pd.read_csv(file, sep='\t', dtype='string', keep_default_na=False, nrows=1000)
        print(kind, Path(file).name, sample.shape)
        display(sample.head(5))
        display(pd.DataFrame({'dtype': sample.dtypes.astype(str),
                              'empty': sample.eq('').sum(),
                              'whitespace_only': (sample.apply(lambda s: s.str.strip().eq('')) & sample.ne('')).sum()}))


ratings ratings-00006.tsv (1000, 35)


,noteId,raterParticipantId,createdAtMillis,version,agree,disagree,helpful,notHelpful,helpfulnessLevel,helpfulOther,...,notHelpfulArgumentativeOrBiased,notHelpfulOffTopic,notHelpfulSpamHarassmentOrAbuse,notHelpfulIrrelevantSources,notHelpfulOpinionSpeculation,notHelpfulNoteNotNeeded,ratedOnTweetId,ratingSourceBucketed,suggestion,suggestionId
0,1970971908511248564,A938FA3B0AD7391892F3C40E399456F5C8B675A7E57A1D...,1758793693079,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,1970600116626026528,DEFAULT,,-1
1,1970971908511248564,92BED4FA1FB7149B3F2B3D160AF9C60F1EA4997671018B...,1758753604497,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,1970600116626026528,DEFAULT,,-1
2,1970971908511248564,67630C972F5AB6E472C7D61B906A03727E12211E697A51...,1758752430191,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,1970600116626026528,DEFAULT,,-1
3,1970971908511248564,B7A8489C697EE2DF936913550F97845B4F8BEACB8E21ED...,1758756352917,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,1970600116626026528,DEFAULT,,-1
4,1970971908511248564,F52241B430D12F73F55BCCC1BFDBFCCC85E002BA860F96...,1758852989121,2,0,0,0,0,HELPFUL,1,...,0,0,0,0,0,0,1970600116626026528,DEFAULT,,-1


,dtype,empty,whitespace_only
noteId,string,0,0
raterParticipantId,string,0,0
createdAtMillis,string,0,0
version,string,0,0
agree,string,0,0
disagree,string,0,0
helpful,string,0,0
notHelpful,string,0,0
helpfulnessLevel,string,0,0
helpfulOther,string,0,0


status noteStatusHistory-00000.tsv (1000, 24)


,noteId,noteAuthorParticipantId,createdAtMillis,timestampMillisOfFirstNonNMRStatus,firstNonNMRStatus,timestampMillisOfCurrentStatus,currentStatus,timestampMillisOfLatestNonNMRStatus,mostRecentNonNMRStatus,timestampMillisOfStatusLock,...,currentGroupStatus,currentDecidedBy,currentModelingGroup,timestampMillisOfMostRecentStatusChange,timestampMillisOfNmrDueToMinStableCrhTime,currentMultiGroupStatus,currentModelingMultiGroup,timestampMinuteOfFinalScoringOutput,timestampMillisOfFirstNmrDueToMinStableCrhTime,timestampMillisAbovePcrhThreshold
0,1352796878438424576,B021517F4F8F9F372DCD70BB8F8A21A80C12D7EFDE8B2F...,1611366884227,,,1721920068424,NEEDS_MORE_RATINGS,,,1674003109873,...,NEEDS_MORE_RATINGS,CoreModel (v1.1),13,-1,,,,29845679,,
1,1353415873227177985,D4C9EDB464DF2D4A6D8F68CA8D1F86226B50EB9D3B0F33...,1611514464087,,,1721920068424,NEEDS_MORE_RATINGS,,,1674003109873,...,NEEDS_MORE_RATINGS,CoreModel (v1.1),13,-1,,,,29845679,,
2,1354586938863443971,C0AF45F4C4B2240E7AB31456957E5D770FF7AFA13627C7...,1611793667892,,,1781032339438,NEEDS_MORE_RATINGS,,,1674003109873,...,,CoreModel (v1.1),,-1,-1,,,29845679,,
3,1354588003075764229,9C480F7B2E882F19F9FBF75BD13FEA53031875AC42FDF6...,1611793921619,,,1721920068424,NEEDS_MORE_RATINGS,,,1674003109873,...,,ExpansionPlusModel (v1.1),,-1,,,,29845679,,
4,1354588172659920899,931B6CFE7C2FDF3653B0A63DC20DF3C51F8AA3BEB9D613...,1611793962051,,,1721920068424,NEEDS_MORE_RATINGS,,,1674003109873,...,,ExpansionPlusModel (v1.1),,-1,,,,29845679,,


,dtype,empty,whitespace_only
noteId,string,0,0
noteAuthorParticipantId,string,0,0
createdAtMillis,string,0,0
timestampMillisOfFirstNonNMRStatus,string,1000,0
firstNonNMRStatus,string,1000,0
timestampMillisOfCurrentStatus,string,0,0
currentStatus,string,0,0
timestampMillisOfLatestNonNMRStatus,string,1000,0
mostRecentNonNMRStatus,string,1000,0
timestampMillisOfStatusLock,string,0,0


## 4. Shared cleaning decisions

- Preserve all original source columns/values, including IDs, text, whitespace and literal labels.
- Remove only exact duplicates across every original field, across shards and chunks. Retain invalid keys and conflicting natural keys with flags.
- Add normalized identifiers, UTC dates, timestamp quality flags and source provenance. Millisecond and minute timestamp fields use their respective units.
- Never impute missing identifiers, times or statuses. A flag alone is not justification for deletion.
- Never interpret unmatched Notes as zero activity; supplied shards have incomplete and unconfirmed release coverage.

### Ratings rules

Require noteId, raterParticipantId, createdAtMillis and helpfulnessLevel. Recognize HELPFUL, SOMEWHAT_HELPFUL and NOT_HELPFUL. Retain other source values and flag them. Check repeated participant-note keys; the observed helpful fraction is not the official Community Notes scoring system.

### Note Status History rules

Require noteId, createdAtMillis, firstNonNMRStatus, timestampMillisOfFirstNonNMRStatus and currentStatus. Check status labels, timestamp consistency and negative latency. Compute preliminary first-non-NMR response hours only for eligible records by the provisional cutoff. Notes creation time must be checked when building the team's final analysis table.


## 5. Execute Ratings 00006 only

This cell executes **ratings-00006.tsv only** using the shared source, with bounded memory. Status History was already cleaned in the earlier run and is not rerun here. The native full-data engine requires an existing C++17 compiler and zlib; set ENGINE to 'pandas' if unavailable (slower). No input files are modified. A successful process creates SUCCESS.json; this marks cleaning completion, not independent validation or complete release coverage.


In [20]:
command = [
    sys.executable, str(WORKFLOW / 'clean_ratings_status.py'),
    '--only', 'ratings',
    '--ratings', *FILES['ratings'],
    '--output', str(OUT),
    '--cutoff', CUTOFF,
    '--engine', ENGINE
]
if PREVIEW_ROWS is not None:
    command += ['--max-rows', str(PREVIEW_ROWS)]
subprocess.run(command, cwd=WORKFLOW, check=True)
success = json.loads((OUT / 'SUCCESS.json').read_text())
print(json.dumps(success, indent=2))


ratings audit scanned 1000000
ratings audit scanned 2000000
ratings audit scanned 3000000
ratings audit scanned 4000000
ratings audit scanned 5000000
ratings audit scanned 6000000
ratings audit scanned 7000000
ratings audit scanned 8000000
ratings audit scanned 9000000
ratings audit scanned 10000000
ratings audit scanned 11000000
ratings audit scanned 12000000
ratings audit scanned 13000000
ratings audit scanned 14000000
ratings audit scanned 15000000
ratings audit scanned 16000000
ratings audit scanned 17000000
ratings audit scanned 18000000
ratings audit scanned 19000000
ratings audit scanned 20000000
ratings audit scanned 21000000
ratings audit scanned 22000000
ratings audit scanned 23000000
ratings audit scanned 24000000
ratings audit scanned 25000000
ratings audit scanned 26000000
ratings audit scanned 27000000
ratings exact audit complete: 27038662 inputs / 27038662 distinct rows
ratings exported 1000000 / 27038662
ratings exported 2000000 / 27038662
ratings exported 3000000 / 27

## 6. Reconcile input, duplicates and output

Each table must satisfy input = output + exact duplicates removed. Scope is based on the configured run, not the inspection prefix. Conflicting rows remain available for review.

In [21]:
reports = {kind: json.loads((OUT / f'{kind}_validation.json').read_text()) for kind in success['reports']}
rows = []
for kind, report in reports.items():
    counts = report['counts']
    assert counts['input'] == counts['output'] + counts.get('exact_duplicates_removed', 0)
    rows.append({'dataset': kind, 'input_rows': counts['input'], 'output_rows': counts['output'],
                 'exact_duplicates_removed': counts.get('exact_duplicates_removed', 0),
                 'invalid_keys_retained': counts.get('invalid_key_retained', 0),
                 'repeated_keys_retained': counts.get('repeated_key_retained', 0)})
display(pd.DataFrame(rows))


,dataset,input_rows,output_rows,exact_duplicates_removed,invalid_keys_retained,repeated_keys_retained
0,ratings,27038662,27038662,0,0,0


## 7. Review quality flags and categories

Counts below describe the actual completed run. The report contains all available QC counts and missingness information. Raw fields are retained alongside additive normalized columns.

In [22]:
for kind, report in reports.items():
    print(kind)
    display(pd.DataFrame([{'check': key, 'count': value} for key, value in report['counts'].items()]))
    print('Category counts:', report.get('categories', {}))
    print('Original-field missingness:', report.get('missing', {}))


ratings


,check,count
0,exact_duplicates_removed,0
1,helpful,16820314
2,input,27038662
3,invalid_key_retained,0
4,invalid_or_missing_createdAtMillis,0
5,output,27038662
6,qc_createdAtMillis_after_cutoff,0
7,qc_createdAtMillis_invalid,0
8,qc_createdAtMillis_missing,0
9,qc_createdAtMillis_utc_unrepresentable,0


Category counts: {'HELPFUL': 16820314, 'NOT_HELPFUL': 9690876, 'SOMEWHAT_HELPFUL': 527472}
Original-field missingness: {'agree': 0, 'createdAtMillis': 0, 'disagree': 0, 'helpful': 0, 'helpfulAddressesClaim': 0, 'helpfulClear': 0, 'helpfulEmpathetic': 0, 'helpfulGoodSources': 0, 'helpfulImportantContext': 0, 'helpfulInformative': 0, 'helpfulOther': 0, 'helpfulUnbiasedLanguage': 0, 'helpfulUniqueContext': 0, 'helpfulnessLevel': 0, 'notHelpful': 0, 'notHelpfulArgumentativeOrBiased': 0, 'notHelpfulHardToUnderstand': 0, 'notHelpfulIncorrect': 0, 'notHelpfulIrrelevantSources': 0, 'notHelpfulMissingKeyPoints': 0, 'notHelpfulNoteNotNeeded': 0, 'notHelpfulOffTopic': 0, 'notHelpfulOpinionSpeculation': 0, 'notHelpfulOpinionSpeculationOrBias': 0, 'notHelpfulOther': 0, 'notHelpfulOutdated': 0, 'notHelpfulSourcesMissingOrUnreliable': 0, 'notHelpfulSpamHarassmentOrAbuse': 0, 'noteId': 0, 'ratedOnTweetId': 0, 'raterParticipantId': 0, 'ratingSourceBucketed': 0, 'suggestion': 27038157, 'suggestionId': 0

## 8. Inspect saved outputs and flagged examples

Load bounded chunks instead of the full Ratings table. At most 20 flagged examples are displayed, scanning at most 100,000 retained rows; this example scan does not establish the total flag count. Refer to the full reports for totals.

In [23]:
for kind in reports:
    path = OUT / f'{kind}_clean.tsv.gz'
    display(pd.read_csv(path, sep='\t', dtype='string', keep_default_na=False, nrows=5))
    examples = []
    scanned = 0
    with pd.read_csv(path, sep='\t', dtype='string', keep_default_na=False, chunksize=10000) as reader:
        for chunk in reader:
            flags = [c for c in chunk if c.startswith('qc_')]
            # Some QC fields express validity/eligibility; False is a concern for validity only.
            concern = pd.Series(False, index=chunk.index)
            for c in flags:
                if any(s in c for s in ['invalid', 'missing', 'unrepresentable', 'after_cutoff', 'negative', 'inconsistent', 'repeated']):
                    concern |= chunk[c].eq('True')
            examples.append(chunk.loc[concern].head(20))
            scanned += len(chunk)
            if sum(len(x) for x in examples) >= 20 or scanned >= 100000:
                break
    print(kind, 'example scan rows:', scanned)
    display(pd.concat(examples, ignore_index=True).head(20))


,noteId,raterParticipantId,createdAtMillis,version,agree,disagree,helpful,notHelpful,helpfulnessLevel,helpfulOther,...,createdAtMillis_utc,qc_createdAtMillis_missing,qc_createdAtMillis_invalid,qc_createdAtMillis_utc_unrepresentable,qc_createdAtMillis_after_cutoff,createdAtUtc,qc_created_by_cutoff,helpfulnessLevel_clean,qc_valid_helpfulness,qc_repeated_key
0,1970971908511248564,A938FA3B0AD7391892F3C40E399456F5C8B675A7E57A1D...,1758793693079,2,0,0,0,0,HELPFUL,0,...,2025-09-25 09:48:13.079000+00:00,False,False,False,False,2025-09-25 09:48:13.079000+00:00,True,HELPFUL,True,False
1,1970971908511248564,92BED4FA1FB7149B3F2B3D160AF9C60F1EA4997671018B...,1758753604497,2,0,0,0,0,HELPFUL,0,...,2025-09-24 22:40:04.497000+00:00,False,False,False,False,2025-09-24 22:40:04.497000+00:00,True,HELPFUL,True,False
2,1970971908511248564,67630C972F5AB6E472C7D61B906A03727E12211E697A51...,1758752430191,2,0,0,0,0,HELPFUL,0,...,2025-09-24 22:20:30.191000+00:00,False,False,False,False,2025-09-24 22:20:30.191000+00:00,True,HELPFUL,True,False
3,1970971908511248564,B7A8489C697EE2DF936913550F97845B4F8BEACB8E21ED...,1758756352917,2,0,0,0,0,HELPFUL,0,...,2025-09-24 23:25:52.917000+00:00,False,False,False,False,2025-09-24 23:25:52.917000+00:00,True,HELPFUL,True,False
4,1970971908511248564,F52241B430D12F73F55BCCC1BFDBFCCC85E002BA860F96...,1758852989121,2,0,0,0,0,HELPFUL,1,...,2025-09-26 02:16:29.121000+00:00,False,False,False,False,2025-09-26 02:16:29.121000+00:00,True,HELPFUL,True,False


ratings example scan rows: 100000


,noteId,raterParticipantId,createdAtMillis,version,agree,disagree,helpful,notHelpful,helpfulnessLevel,helpfulOther,...,createdAtMillis_utc,qc_createdAtMillis_missing,qc_createdAtMillis_invalid,qc_createdAtMillis_utc_unrepresentable,qc_createdAtMillis_after_cutoff,createdAtUtc,qc_created_by_cutoff,helpfulnessLevel_clean,qc_valid_helpfulness,qc_repeated_key


## 9. Independently verify saved data

The verifier checks source/output fields, provenance, row reconciliation and QC, including proof of exact duplicate removals. Full verification scans the configured input/output and may take substantial time. This cell does not claim agreement with Peyton snapshot or platform-wide completeness.

In [24]:
subprocess.run([sys.executable, str(WORKFLOW / 'validate_outputs.py'), str(OUT)],
               cwd=WORKFLOW, check=True)
validation = json.loads((OUT / 'independent_validation.json').read_text())
print(json.dumps(validation, indent=2))


ratings verified all fields and timestamp QC for 1000000 source rows
ratings verified all fields and timestamp QC for 2000000 source rows
ratings verified all fields and timestamp QC for 3000000 source rows
ratings verified all fields and timestamp QC for 4000000 source rows
ratings verified all fields and timestamp QC for 5000000 source rows
ratings verified all fields and timestamp QC for 6000000 source rows
ratings verified all fields and timestamp QC for 7000000 source rows
ratings verified all fields and timestamp QC for 8000000 source rows
ratings verified all fields and timestamp QC for 9000000 source rows
ratings verified all fields and timestamp QC for 10000000 source rows
ratings verified all fields and timestamp QC for 11000000 source rows
ratings verified all fields and timestamp QC for 12000000 source rows
ratings verified all fields and timestamp QC for 13000000 source rows
ratings verified all fields and timestamp QC for 14000000 source rows
ratings verified all fields a

## 10. Team integration with Peyton

Read Peyton's notes_clean.csv with dtype='string', keep_default_na=False. Notes noteId must be unique before a Ratings many:1 or Status 1:1 join. Aggregate Ratings per note before building one row per note; do not merge raw Ratings into a one-row-per-note table without accounting for multiplication. Note Requests connect through tweetId; aggregate/check cardinality separately.

Peyton's agreed Notes scope is notes-00001.tsv only. Do not silently extend it with 00002. Our local reproduction of the supplied source produced 12,304 Notes and 487,851 Requests; Peyton's pasted executed notebook reports 15,430 and 499,888. **These are different record counts: source snapshot/hash equality must be confirmed before combining dataset findings.** This difference does not establish a code error.

The local compatibility check matched 9,560 Status rows with no creation-time discrepancies, and 89,364 Ratings records to those locally reproduced Notes. These are prior audited local results, not outputs from this notebook execution. Original values and exact-dedup policies are compatible; timestamp edge-case and QC conventions differ. Common cutoff remains unconfirmed.


## 11. Verified single-file Ratings results and retained Status result

Final input scope is ratings-00006.tsv plus noteStatusHistory-00000.tsv. Ratings 00007 is excluded from this submission. The saved code outputs above independently verify all 27,038,662 Ratings records from 00006: PASS, no exact duplicates, invalid keys or repeated participant-note keys. All original fields were preserved.

| Dataset | Input rows | Exact duplicates removed | Retained rows |
|---|---:|---:|---:|
| Ratings 00006 only — current verified run | 27,038,662 | 0 | 27,038,662 |
| Status History 00000 — earlier completed cleaning run | 3,343,818 | 0 | 3,343,818 |

Current Ratings output: outputs/full/20261018T050534418856Z/ratings_clean.tsv.gz.
Retained project Status output: outputs/full/20261018T043052797164Z/status_clean.tsv.gz. The earlier project combined verification was interrupted; do not describe that combined run as independently verified. An independent full Status validation from the original delivered package is preserved separately in evidence/status_prior_independent_validation.json; it is not a newly performed validation of this project copy.

The earlier 53,935,477-row input count and 23,916 duplicate removals applied to two Ratings shards and are superseded for this submission. Do not reuse them as single-file findings. Coverage is all rows of the specified files; platform-wide completeness and team snapshot/cutoff agreement remain unverified.

## 12. Submission checklist

Save the executed notebook, shared source, README and compact reports. Keep large cleaned datasets outside Git. Retain Status and Ratings output paths distinctly; no raw Ratings-to-Notes join is claimed in this single-file run. The locally reproduced Peyton Notes dataset differed in row counts from the supplied executed Peyton notebook, so confirm input hashes/snapshots before combining findings. No remote main merge has been performed.
